
# KA SS 26 · Challenge 2 — Stage 13
## DINOv3 ViT-S/16 + high-resolution CNN refinement + foreground/seed watershed

This is the deliberately **different final architecture path**.

### Why DINOv3 instead of DINOv2
The primary encoder is **DINOv3 ViT-S/16** (`vit_small_patch16_dinov3.lvd1689m`, 21.6M parameters). Meta describes DINOv3 as producing high-quality dense features and it is directly supported in recent `timm`. DINOv2 ViT-S/14 remains an explicit fallback only if the DINOv3 weights cannot be loaded.

### Architecture
```text
DINOv3 ViT-S/16 intermediate features (blocks 2, 5, 8, 11)
            │
            ├── multi-layer transformer fusion at 1/16 resolution
            │
input ── shallow CNN detail branch at 1x / 1/2 / 1/4 / 1/8
            │
            └── progressive decoder with learned high-resolution refinement
                        ↓
              foreground + e70 seed maps
                        ↓
             marker-controlled watershed
                        ↓
                  instance masks
```

The shallow CNN branch is intentional: a patch-16 transformer alone is too coarse for cells whose lower-tail size is around 11 px.

### Dense-image strategy
Stage 10's audit showed count calibration is good on sparse/medium images and degrades with density. This notebook therefore uses:
- **relative seed erosion = 0.70**
- **D4 / 90° symmetry**
- **density-biased scale augmentation before cropping**
- **dense-image oversampling**
- nearest-neighbor resizing for the seed channel

### Important rules note
DINOv3 ViT-S/16 is pretrained on **generic web imagery (LVD-1689M)**, not cell/microscopy/medical data. The competition guidance we have says generic pretrained weights are allowed, but DINOv3 was not specifically TA-confirmed the way ImageNet/COCO were. **Run the experiment now; only submit the CSV if the TA confirms this generic DINOv3 pretraining is acceptable.**

### Runtime
Designed for **Google Colab T4**, one model, mixed precision, gradient checkpointing, no ensemble. Expected end-to-end runtime is roughly **1.5–3 hours**.

### Outputs
- `/content/13th_submission.csv`
- `/content/13th_REPORT.md`
- Drive checkpoint/maps under `MyDrive/ka_ss26/stage13_dinov3/`


In [2]:

# 1 · GPU, packages, imports
import os, sys, subprocess, time, math, random, gc, json, shutil, zipfile, warnings
from pathlib import Path
from collections import defaultdict

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip() or "NO GPU DETECTED")

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs], check=True)

pip_install("timm>=1.0.20", "kaggle", "scikit-image", "scikit-learn")

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from scipy import ndimage as ndi
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from skimage.segmentation import watershed
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Enable a GPU runtime in Colab before continuing."
torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

T0 = time.time()
def elapsed():
    s = time.time() - T0
    return f"{int(s//3600)}h {int((s%3600)//60):02d}m {int(s%60):02d}s"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

print("torch:", torch.__version__)
print("timm :", timm.__version__)
print("device:", DEVICE)


Tesla T4, 15360 MiB, 580.82.07
torch: 2.11.0+cu128
timm : 1.0.28
device: cuda


In [3]:

# 2 · Configuration
class CFG:
    SEED = 43
    RUN_TAG = "stage13_dinov3_vits16"
    SUB_INDEX_COL = "idx"

    IMG_H, IMG_W = 520, 704
    DUP_CORR_THR = 0.72
    N_FOLDS = 5
    FOLD = 0

    PRIMARY_ENCODER = "vit_small_patch16_dinov3.lvd1689m"
    FALLBACK_ENCODER = "vit_small_patch14_dinov2.lvd142m"
    ALLOW_DINOV2_FALLBACK = True
    INTERMEDIATE_BLOCKS = (2, 5, 8, 11)

    SEED_EROSION_FRAC = 0.70
    BORDER_DILATION = 2
    BORDER_WEIGHT = 4.0

    CROP_H, CROP_W = 400, 512
    EPOCHS = 55
    STEPS_PER_EPOCH = 42
    BATCH_SIZE = 4
    DECODER_LR = 2.0e-4
    ENCODER_LR = 2.0e-5
    WEIGHT_DECAY = 1.0e-4
    FREEZE_ENCODER_EPOCHS = 3
    WARMUP_STEPS = 120
    GRAD_CLIP = 5.0
    AMP = True
    NUM_WORKERS = 2
    EVAL_START = 15
    EVAL_EVERY = 5
    QUICK_VAL_N = 86

    SCALE_MIX = (
        (0.45, 0.70, 0.84),
        (0.35, 0.86, 1.00),
        (0.20, 1.00, 1.16),
    )
    DENSE_GT_THR = 200
    DENSE_REPEAT = 2
    AUG_HFLIP = 0.5
    AUG_VFLIP = 0.5
    AUG_BRIGHT = 0.06
    AUG_CONTRAST = 0.12
    AUG_NOISE = 0.010
    AUG_BLUR_P = 0.12

    DEFAULT_POST = dict(t_cell=0.50, t_seed=0.65, min_area=90, score_thr=0.0)

    TC_GRID = (0.40, 0.45, 0.50, 0.55, 0.60)
    TS_GRID = (0.50, 0.55, 0.60, 0.65, 0.70, 0.75)
    MA_GRID = (30, 50, 70, 90, 110)
    PLATEAU_TOL = 0.004

    TTA_FLIPS = True
    SUBMISSION_RULES_CONFIRMED = False

seed_everything(CFG.SEED)
STATS = {}
print("primary encoder:", CFG.PRIMARY_ENCODER)
print("epochs:", CFG.EPOCHS, "| steps/epoch:", CFG.STEPS_PER_EPOCH)


primary encoder: vit_small_patch16_dinov3.lvd1689m
epochs: 55 | steps/epoch: 42



## 3 · Data access through Kaggle API

Put `kaggle.json` at `/content/drive/MyDrive/ka_ss26/kaggle.json`, as in the earlier Colab stages.


In [5]:
# 3A · Mount Drive + configure Kaggle + download/reuse competition data

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
KAGGLE_JSON = Path("/content/kaggle.json")
KAGGLE_COMPETITION = "ka-ss-26-challenge-2"

DATA_ROOT = Path("/content/data")

# Expensive Stage 13 artifacts are preserved in Drive
WORK = Path("/content/drive/MyDrive/ka_ss26/stage13_dinov3")
ART  = WORK / "artifacts"
CKPT = WORK / "ckpt"
MAPS = WORK / "maps"

for p in (DATA_ROOT, WORK, ART, CKPT, MAPS):
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Kaggle credentials
# ------------------------------------------------------------------
assert KAGGLE_JSON.is_file(), (
    "kaggle.json not found at /content/kaggle.json\n"
    "Upload kaggle.json to the Colab Files panel, then rerun this cell."
)

KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    KAGGLE_JSON,
    KAGGLE_DIR / "kaggle.json",
)

(KAGGLE_DIR / "kaggle.json").chmod(0o600)

print(">>> Kaggle credentials configured")
print("source:", KAGGLE_JSON)
print("installed:", KAGGLE_DIR / "kaggle.json")


# ------------------------------------------------------------------
# Locate an already-extracted copy if this cell was run previously
# ------------------------------------------------------------------
def looks_like_dataset(p):
    return (
        (p / "train.csv").is_file()
        and (p / "train").is_dir()
        and (p / "test").is_dir()
    )


def find_dataset_root(base):
    if looks_like_dataset(base):
        return base

    for csv_path in base.rglob("train.csv"):
        candidate = csv_path.parent

        if looks_like_dataset(candidate):
            return candidate

    return None


BASE = find_dataset_root(DATA_ROOT)


# ------------------------------------------------------------------
# Download only if the dataset is not already present
# ------------------------------------------------------------------
if BASE is None:

    print("\n>>> Competition data not found locally")
    print(">>> Downloading from Kaggle...")

    result = subprocess.run(
        [
            "kaggle",
            "competitions",
            "download",
            "-c",
            KAGGLE_COMPETITION,
            "-p",
            str(DATA_ROOT),
            "--force",
        ],
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)

        raise RuntimeError(
            "Kaggle competition download failed. "
            "Check that you joined the competition and accepted its rules."
        )

    # Extract every downloaded archive
    zip_files = list(DATA_ROOT.glob("*.zip"))

    assert zip_files, (
        "Kaggle command completed but no ZIP file was found "
        f"inside {DATA_ROOT}"
    )

    for zip_path in zip_files:
        print(">>> Extracting:", zip_path.name)

        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DATA_ROOT)

    BASE = find_dataset_root(DATA_ROOT)


# ------------------------------------------------------------------
# Final dataset validation
# ------------------------------------------------------------------
assert BASE is not None, (
    "Competition files were downloaded/extracted, "
    "but the train/test dataset structure could not be located."
)

TRAIN_DIR = BASE / "train"
TEST_DIR  = BASE / "test"
TRAIN_CSV = BASE / "train.csv"

assert TRAIN_DIR.is_dir()
assert TEST_DIR.is_dir()
assert TRAIN_CSV.is_file()

TRAIN_IDS = sorted(
    p.stem
    for p in TRAIN_DIR.glob("*.png")
)

TEST_IDS = sorted(
    p.stem
    for p in TEST_DIR.glob("*.png")
)

assert TRAIN_IDS, "No training PNG files found."
assert TEST_IDS, "No test PNG files found."

with Image.open(
    TRAIN_DIR / f"{TRAIN_IDS[0]}.png"
) as im:
    CFG.IMG_W, CFG.IMG_H = im.size

NPIX = CFG.IMG_H * CFG.IMG_W


# ------------------------------------------------------------------
# Competition sanity checks
# ------------------------------------------------------------------
print("\n>>> Dataset ready")
print("dataset root:", BASE)
print("train images:", len(TRAIN_IDS))
print("test images :", len(TEST_IDS))
print("image HxW   :", CFG.IMG_H, "x", CFG.IMG_W)
print("\n>>> Stage 13 artifacts will be saved to:")
print(WORK)

assert len(TRAIN_IDS) == 424, (
    f"Expected 424 training images, found {len(TRAIN_IDS)}"
)

assert len(TEST_IDS) == 182, (
    f"Expected 182 test images, found {len(TEST_IDS)}"
)

print("\n>>> Stage 13 data setup passed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> Kaggle credentials configured
source: /content/kaggle.json
installed: /root/.kaggle/kaggle.json

>>> Competition data not found locally
>>> Downloading from Kaggle...


>>> Extracting: ka-ss-26-challenge-2.zip

>>> Dataset ready
dataset root: /content/data/cells_segmentation
train images: 424
test images : 182
image HxW   : 520 x 704

>>> Stage 13 artifacts will be saved to:
/content/drive/MyDrive/ka_ss26/stage13_dinov3

>>> Stage 13 data setup passed


In [6]:

# 4 · Schema, RLE, GT cache, official metric
df_raw = pd.read_csv(TRAIN_CSV)
ID_COL = next(c for c in df_raw.columns if c.lower() in ("id","image_id","imageid"))
ANN_COL = next(c for c in df_raw.columns if c.lower() in ("annotation","encodedpixels","rle","segmentation"))
print("columns:", df_raw.columns.tolist())
print("ID:", ID_COL, "| RLE:", ANN_COL)

def rle_decode_flat(rle, h, w, order="C"):
    if not isinstance(rle, str) or not rle.strip() or rle.strip() in ("-1","nan"):
        return np.empty(0, np.int64)
    v = np.asarray(rle.split(), np.int64)
    starts, lengths = v[0::2] - 1, v[1::2]
    idx = np.concatenate([np.arange(s, s+l) for s,l in zip(starts,lengths)])
    if order == "C":
        return idx
    return (idx % h) * w + (idx // h)

def rle_encode(mask_bool):
    p = np.concatenate([[0], mask_bool.reshape(-1).astype(np.uint8), [0]])
    runs = np.where(p[1:] != p[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(int(x)) for x in runs)

def flat_to_rle(flat):
    m = np.zeros(NPIX, np.uint8)
    m[np.asarray(flat, np.int64)] = 1
    return rle_encode(m)

probe = df_raw[df_raw[ANN_COL].astype(str) != "-1"].sample(min(200, len(df_raw)), random_state=0)
quality = {}
for order in ("C","F"):
    one = []
    for r in probe[ANN_COL].astype(str):
        f = rle_decode_flat(r, CFG.IMG_H, CFG.IMG_W, order)
        if f.size == 0 or f.max() >= NPIX:
            continue
        m = np.zeros(NPIX, np.uint8); m[f] = 1
        one.append(ndi.label(m.reshape(CFG.IMG_H, CFG.IMG_W))[1] == 1)
    quality[order] = float(np.mean(one))
RLE_ORDER = max(quality, key=quality.get)
print("RLE order:", RLE_ORDER, quality)

INST = defaultdict(list)
for image_id, rle in zip(df_raw[ID_COL].astype(str), df_raw[ANN_COL].astype(str)):
    f = rle_decode_flat(rle, CFG.IMG_H, CFG.IMG_W, RLE_ORDER)
    if f.size:
        INST[image_id].append(f.astype(np.int32))

gt_counts = np.array([len(INST[i]) for i in TRAIN_IDS])
areas = np.array([len(f) for i in TRAIN_IDS for f in INST[i]])
print("GT instances/image mean/median/p95/max:",
      round(float(gt_counts.mean()),1), int(np.median(gt_counts)),
      int(np.percentile(gt_counts,95)), int(gt_counts.max()))

def sparse_masks(flats):
    if not flats:
        return csr_matrix((0, NPIX), dtype=np.float32)
    indptr = np.zeros(len(flats)+1, np.int64)
    indptr[1:] = np.cumsum([len(f) for f in flats])
    indices = np.concatenate(flats).astype(np.int64)
    return csr_matrix((np.ones(indices.size, np.float32), indices, indptr),
                      shape=(len(flats), NPIX))

def dice_matrix(gt, pred):
    if len(gt)==0 or len(pred)==0:
        return np.zeros((len(gt),len(pred)), np.float32)
    a, b = sparse_masks(gt), sparse_masks(pred)
    inter = (a @ b.T).toarray()
    aa = np.asarray(a.sum(axis=1)).reshape(-1,1)
    bb = np.asarray(b.sum(axis=1)).reshape(1,-1)
    return (2*inter / np.maximum(aa+bb,1)).astype(np.float32)

def instance_f1(gt, pred, thr=0.50):
    d = dice_matrix(gt, pred)
    ng, npred = d.shape
    if ng==0 and npred==0: return 1.0
    if ng==0 or npred==0: return 0.0
    gi, pi = np.nonzero(d >= thr)
    if gi.size == 0: return 0.0
    order = np.argsort(-d[gi,pi])
    ug, up, tp = set(), set(), 0
    for g,p in zip(gi[order], pi[order]):
        if int(g) in ug or int(p) in up: continue
        ug.add(int(g)); up.add(int(p)); tp += 1
    return 2*tp / (ng + npred)

x = np.array([0,1,2], np.int32)
assert abs(instance_f1([x],[x])-1.0) < 1e-8
assert abs(instance_f1([x],[x,x]) - 2/3) < 1e-8
print("official metric tests passed")


columns: ['Unnamed: 0', 'id', 'annotation']
ID: id | RLE: annotation
RLE order: C {'C': 0.915, 'F': 0.0}
GT instances/image mean/median/p95/max: 125.2 47 503 782
official metric tests passed


In [7]:

# 5 · Image cache + grouped split
IMAGES = {}
thumbs = []
for image_id in TRAIN_IDS:
    arr = np.array(Image.open(TRAIN_DIR/f"{image_id}.png").convert("L"), np.uint8)
    IMAGES[image_id] = arr
    th = cv2.resize(arr, (32,32), interpolation=cv2.INTER_AREA).astype(np.float32)
    th = (th - th.mean()) / (th.std() + 1e-6)
    thumbs.append(th.reshape(-1))

TH = np.stack(thumbs)
CORR = np.corrcoef(TH)
adj = (CORR >= CFG.DUP_CORR_THR).astype(np.uint8)
np.fill_diagonal(adj, 1)
n_groups, group_labels = connected_components(csr_matrix(adj), directed=False)
print("correlation groups:", n_groups)

gkf = GroupKFold(n_splits=CFG.N_FOLDS)
splits = list(gkf.split(np.arange(len(TRAIN_IDS)), groups=group_labels))
tr_idx, va_idx = splits[CFG.FOLD]
train_ids = [TRAIN_IDS[i] for i in tr_idx]
val_ids = [TRAIN_IDS[i] for i in va_idx]

cross = CORR[np.ix_(va_idx, tr_idx)]
print("train/val:", len(train_ids), len(val_ids))
print("max val->train corr:", round(float(cross.max()),3))

GT_COUNT = {i: len(INST[i]) for i in TRAIN_IDS}
dense_ids = [i for i in train_ids if GT_COUNT[i] >= CFG.DENSE_GT_THR]
train_ids_boost = list(train_ids) + dense_ids * CFG.DENSE_REPEAT
print("dense train images:", len(dense_ids), "/", len(train_ids))
print("boosted ID list:", len(train_ids_boost))

STATS.update(groups=int(n_groups), n_train=len(train_ids), n_val=len(val_ids),
             max_val_train_corr=float(cross.max()))


correlation groups: 424
train/val: 339 85
max val->train corr: 0.666
dense train images: 77 / 339
boosted ID list: 493


In [8]:

# 6 · e70 targets
def build_targets(image_id):
    h,w = CFG.IMG_H, CFG.IMG_W
    foreground = np.zeros((h,w), np.uint8)
    seed = np.zeros((h,w), np.uint8)
    overlap = np.zeros((h,w), np.uint16)
    pad = CFG.BORDER_DILATION + 1

    for flat in INST[image_id]:
        ys,xs = np.divmod(flat.astype(np.int64), w)
        y0,y1 = max(0,ys.min()-pad), min(h,ys.max()+1+pad)
        x0,x1 = max(0,xs.min()-pad), min(w,xs.max()+1+pad)
        patch = np.zeros((y1-y0,x1-x0), np.uint8)
        patch[ys-y0,xs-x0] = 1
        foreground[y0:y1,x0:x1] |= patch

        dist = ndi.distance_transform_edt(patch)
        core = (dist > max(1.0, CFG.SEED_EROSION_FRAC * dist.max())).astype(np.uint8)
        if not core.any():
            yy,xx = np.unravel_index(np.argmax(dist), dist.shape)
            core[yy,xx] = 1
        seed[y0:y1,x0:x1] |= core

        overlap[y0:y1,x0:x1] += ndi.binary_dilation(
            patch, iterations=CFG.BORDER_DILATION
        ).astype(np.uint16)

    border = (overlap > 1).astype(np.uint8)
    return np.stack([foreground, seed, border])

TARGETS = {i: build_targets(i) for i in TRAIN_IDS}
print("seed coverage:", round(100*np.mean([TARGETS[i][1].mean() for i in TRAIN_IDS]),3), "%")


seed coverage: 1.558 %


In [9]:

# 7 · Density-focused augmentation + dataset
MEAN = np.array([0.485,0.456,0.406], np.float32)[:,None,None]
STD = np.array([0.229,0.224,0.225], np.float32)[:,None,None]

def sample_scale():
    r, acc = random.random(), 0.0
    for p,lo,hi in CFG.SCALE_MIX:
        acc += p
        if r < acc:
            return random.uniform(lo,hi)
    _,lo,hi = CFG.SCALE_MIX[-1]
    return random.uniform(lo,hi)

def resize_full(img, tgt, s):
    h,w = img.shape
    nh,nw = max(64,int(round(h*s))), max(64,int(round(w*s)))
    img_r = cv2.resize(img,(nw,nh),interpolation=cv2.INTER_LINEAR)
    fg = cv2.resize(tgt[0].astype(np.float32),(nw,nh),
                    interpolation=cv2.INTER_AREA if s<1 else cv2.INTER_NEAREST)
    fg = (fg > 0.5).astype(np.uint8)
    seed = cv2.resize(tgt[1],(nw,nh),interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    border = cv2.resize(tgt[2],(nw,nh),interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    return img_r, np.stack([fg,seed,border])

def pad_to_crop(img, tgt, ch, cw):
    h,w = img.shape
    ph,pw = max(0,ch-h), max(0,cw-w)
    if ph or pw:
        img = np.pad(img, ((0,ph),(0,pw)), mode="reflect")
        tgt = np.pad(tgt, ((0,0),(0,ph),(0,pw)), mode="constant")
    return img,tgt

class CellDataset(Dataset):
    def __init__(self, ids, train=True):
        self.ids = list(ids)
        self.train = train
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        image_id = self.ids[idx]
        img = IMAGES[image_id].astype(np.float32) / 255.0
        tgt = TARGETS[image_id].copy()

        if self.train:
            k = random.randint(0,3)
            if k:
                img = np.rot90(img,k)
                tgt = np.rot90(tgt,k,axes=(1,2))
            if random.random() < CFG.AUG_HFLIP:
                img,tgt = img[:,::-1], tgt[:,:,::-1]
            if random.random() < CFG.AUG_VFLIP:
                img,tgt = img[::-1,:], tgt[:,::-1,:]
            img,tgt = np.ascontiguousarray(img), np.ascontiguousarray(tgt)

            s = sample_scale()
            if abs(s-1.0) > 1e-3:
                img,tgt = resize_full(img,tgt,s)

            img,tgt = pad_to_crop(img,tgt,CFG.CROP_H,CFG.CROP_W)
            h,w = img.shape
            y0 = random.randint(0,h-CFG.CROP_H)
            x0 = random.randint(0,w-CFG.CROP_W)
            img = img[y0:y0+CFG.CROP_H, x0:x0+CFG.CROP_W]
            tgt = tgt[:,y0:y0+CFG.CROP_H, x0:x0+CFG.CROP_W]

            img = img + random.uniform(-CFG.AUG_BRIGHT,CFG.AUG_BRIGHT)
            img = (img-0.5)*(1+random.uniform(-CFG.AUG_CONTRAST,CFG.AUG_CONTRAST))+0.5
            if random.random() < CFG.AUG_BLUR_P:
                img = cv2.GaussianBlur(img,(3,3),random.uniform(0.3,0.8))
            img = img + np.random.randn(*img.shape).astype(np.float32)*CFG.AUG_NOISE
            img = np.clip(img,0,1)

        x = np.repeat(img[None],3,axis=0).astype(np.float32)
        x = (x-MEAN)/STD
        y = tgt[:2].astype(np.float32)
        w = 1.0 + CFG.BORDER_WEIGHT*tgt[2].astype(np.float32)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy()), torch.from_numpy(w.copy()), image_id

ds = CellDataset(train_ids_boost, True)
x,y,w,_ = ds[0]
print("sample:", tuple(x.shape), tuple(y.shape), tuple(w.shape))
assert x.shape[-2:] == (CFG.CROP_H,CFG.CROP_W)


sample: (3, 400, 512) (2, 400, 512) (400, 512)


In [10]:

# 8 · DINOv3 encoder + high-resolution refinement decoder
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin,cout,3,stride=stride,padding=1,bias=False),
            nn.BatchNorm2d(cout), nn.GELU(),
            nn.Conv2d(cout,cout,3,padding=1,bias=False),
            nn.BatchNorm2d(cout), nn.GELU(),
        )
    def forward(self,x):
        return self.net(x)

class Stage13DINO(nn.Module):
    def __init__(self, encoder_name, pretrained=True):
        super().__init__()
        self.encoder_name = encoder_name
        self.encoder = timm.create_model(encoder_name, pretrained=pretrained, num_classes=0)
        if hasattr(self.encoder, "set_grad_checkpointing"):
            self.encoder.set_grad_checkpointing(True)

        was_training = self.encoder.training
        self.encoder.eval()
        with torch.no_grad():
            dummy = torch.zeros(1,3,256,256)
            feats = self.encoder.forward_intermediates(
                dummy, indices=CFG.INTERMEDIATE_BLOCKS, intermediates_only=True
            )
        if was_training:
            self.encoder.train()
        feats = list(feats)
        assert len(feats) == 4, f"Expected 4 DINO feature maps, got {len(feats)}"
        assert all(f.ndim == 4 for f in feats), [f.shape for f in feats]
        channels = [f.shape[1] for f in feats]
        print("DINO intermediate shapes:", [tuple(f.shape) for f in feats])

        self.proj = nn.ModuleList([nn.Conv2d(ci,64,1) for ci in channels])
        self.fuse = ConvBlock(256,192)

        self.d1 = ConvBlock(3,32,1)
        self.d2 = ConvBlock(32,48,2)
        self.d3 = ConvBlock(48,64,2)
        self.d4 = ConvBlock(64,96,2)

        self.up8 = ConvBlock(192+96,160)
        self.up4 = ConvBlock(160+64,128)
        self.up2 = ConvBlock(128+48,96)
        self.up1 = ConvBlock(96+32,64)
        self.head = nn.Sequential(ConvBlock(64,48), nn.Conv2d(48,2,1))

    def forward(self,x):
        feats = list(self.encoder.forward_intermediates(
            x, indices=CFG.INTERMEDIATE_BLOCKS, intermediates_only=True
        ))
        z = self.fuse(torch.cat([p(f) for p,f in zip(self.proj,feats)],dim=1))

        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)

        z = F.interpolate(z,size=d4.shape[-2:],mode="bilinear",align_corners=False)
        z = self.up8(torch.cat([z,d4],1))
        z = F.interpolate(z,size=d3.shape[-2:],mode="bilinear",align_corners=False)
        z = self.up4(torch.cat([z,d3],1))
        z = F.interpolate(z,size=d2.shape[-2:],mode="bilinear",align_corners=False)
        z = self.up2(torch.cat([z,d2],1))
        z = F.interpolate(z,size=d1.shape[-2:],mode="bilinear",align_corners=False)
        z = self.up1(torch.cat([z,d1],1))
        return self.head(z)

def load_stage13_model():
    try:
        print("loading DINOv3:", CFG.PRIMARY_ENCODER)
        m = Stage13DINO(CFG.PRIMARY_ENCODER, pretrained=True)
        return m, CFG.PRIMARY_ENCODER, "DINOv3"
    except Exception as e:
        print("\nDINOv3 LOAD FAILED:", type(e).__name__, str(e)[:500])
        if not CFG.ALLOW_DINOV2_FALLBACK:
            raise
        print("\n>>> EXPLICIT FALLBACK TO DINOv2:", CFG.FALLBACK_ENCODER)
        m = Stage13DINO(CFG.FALLBACK_ENCODER, pretrained=True)
        return m, CFG.FALLBACK_ENCODER, "DINOv2-fallback"

model, ACTIVE_ENCODER, ACTIVE_FAMILY = load_stage13_model()
model = model.to(DEVICE)
with torch.no_grad(), torch.autocast("cuda",dtype=torch.float16,enabled=CFG.AMP):
    out = model(torch.randn(1,3,CFG.CROP_H,CFG.CROP_W,device=DEVICE))
print("model output:", tuple(out.shape), "| active:", ACTIVE_FAMILY, ACTIVE_ENCODER)
assert out.shape == (1,2,CFG.CROP_H,CFG.CROP_W)
del out
torch.cuda.empty_cache()


loading DINOv3: vit_small_patch16_dinov3.lvd1689m


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.4MB            

model.safetensors: downloading bytes:           |  0.00B            

DINO intermediate shapes: [(1, 384, 16, 16), (1, 384, 16, 16), (1, 384, 16, 16), (1, 384, 16, 16)]
model output: (1, 2, 400, 512) | active: DINOv3 vit_small_patch16_dinov3.lvd1689m


In [11]:

# 9 · Loss, watershed, inference
def dice_loss_logits(logits, target):
    p = torch.sigmoid(logits)
    dims = (0,2,3)
    inter = (p*target).sum(dims)
    den = p.sum(dims)+target.sum(dims)
    return (1 - (2*inter+1)/(den+1)).mean()

def loss_fn(logits, target, weight):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    bce[:,0] *= weight
    return bce.mean() + dice_loss_logits(logits,target)

def instances_from_maps(fg, seed, post):
    cell = fg >= post["t_cell"]
    markers,nm = ndi.label(seed >= post["t_seed"])
    if nm == 0 or not cell.any():
        return []
    lab = watershed(-fg, markers, mask=cell)
    out = []
    for k in range(1,int(lab.max())+1):
        flat = np.flatnonzero(lab.reshape(-1)==k).astype(np.int32)
        if flat.size >= post["min_area"]:
            out.append(flat)
    return out

def pad16(arr):
    h,w = arr.shape
    ph = (16-h%16)%16
    pw = (16-w%16)%16
    return np.pad(arr,((0,ph),(0,pw)),mode="reflect"), (h,w)

def input_tensor(gray):
    a,orig = pad16(gray.astype(np.float32)/255.0)
    x = np.repeat(a[None],3,axis=0)
    x = (x-MEAN)/STD
    return torch.from_numpy(x[None].copy()).to(DEVICE), orig

@torch.inference_mode()
def predict_one(m, gray, tta=False):
    m.eval()
    x, (h,w) = input_tensor(gray)
    views = [(x, lambda z:z)]
    if tta:
        views = [
            (x, lambda z:z),
            (torch.flip(x,[-1]), lambda z:torch.flip(z,[-1])),
            (torch.flip(x,[-2]), lambda z:torch.flip(z,[-2])),
            (torch.flip(x,[-2,-1]), lambda z:torch.flip(z,[-2,-1])),
        ]
    acc = None
    for xv,inv in views:
        with torch.autocast("cuda",dtype=torch.float16,enabled=CFG.AMP):
            p = torch.sigmoid(m(xv))
        p = inv(p).float()
        acc = p if acc is None else acc+p
    p = (acc/len(views))[0,:,:h,:w].cpu().numpy()
    return p.astype(np.float32)

@torch.inference_mode()
def quick_val_f1(m, ids, post):
    vals=[]
    for image_id in ids:
        p = predict_one(m, IMAGES[image_id], tta=False)
        pred = instances_from_maps(p[0],p[1],post)
        vals.append(instance_f1(INST[image_id],pred))
    return float(np.mean(vals))


In [12]:

# 10 · Train one DINO model
BEST_PATH = CKPT / f"{CFG.RUN_TAG}_best.pt"
TRAIN_LOG = ART / f"{CFG.RUN_TAG}_train.csv"

enc_params = list(model.encoder.parameters())
enc_ids = {id(p) for p in enc_params}
dec_params = [p for p in model.parameters() if id(p) not in enc_ids]

opt = torch.optim.AdamW(
    [{"params":dec_params,"lr":CFG.DECODER_LR},
     {"params":enc_params,"lr":CFG.ENCODER_LR}],
    weight_decay=CFG.WEIGHT_DECAY
)
scaler = torch.amp.GradScaler("cuda",enabled=CFG.AMP)

loader = DataLoader(
    CellDataset(train_ids_boost,True),
    batch_size=CFG.BATCH_SIZE, shuffle=True,
    num_workers=CFG.NUM_WORKERS, pin_memory=True,
    drop_last=True, persistent_workers=CFG.NUM_WORKERS>0
)

total_steps = CFG.EPOCHS * CFG.STEPS_PER_EPOCH
def lr_mult(step):
    if step < CFG.WARMUP_STEPS:
        return (step+1)/CFG.WARMUP_STEPS
    p = (step-CFG.WARMUP_STEPS)/max(1,total_steps-CFG.WARMUP_STEPS)
    return 0.5*(1+math.cos(math.pi*min(1.0,p)))

best_f1, best_epoch = -1.0, -1
history = []
step = 0
it = iter(loader)

for epoch in range(1,CFG.EPOCHS+1):
    freeze = epoch <= CFG.FREEZE_ENCODER_EPOCHS
    for p in enc_params:
        p.requires_grad_(not freeze)

    model.train()
    running = 0.0
    for _ in range(CFG.STEPS_PER_EPOCH):
        try:
            x,y,w,_ = next(it)
        except StopIteration:
            it = iter(loader)
            x,y,w,_ = next(it)

        x=x.to(DEVICE,non_blocking=True)
        y=y.to(DEVICE,non_blocking=True)
        w=w.to(DEVICE,non_blocking=True)

        mult=lr_mult(step)
        opt.param_groups[0]["lr"]=CFG.DECODER_LR*mult
        opt.param_groups[1]["lr"]=0.0 if freeze else CFG.ENCODER_LR*mult

        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16,enabled=CFG.AMP):
            logits=model(x)
            loss=loss_fn(logits,y,w)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(),CFG.GRAD_CLIP)
        scaler.step(opt); scaler.update()
        running += float(loss.item())
        step += 1

    ep_loss = running/CFG.STEPS_PER_EPOCH
    msg=f"ep {epoch:02d}/{CFG.EPOCHS} loss {ep_loss:.4f}"

    if epoch>=CFG.EVAL_START and ((epoch-CFG.EVAL_START)%CFG.EVAL_EVERY==0 or epoch==CFG.EPOCHS):
        q=quick_val_f1(model,val_ids[:CFG.QUICK_VAL_N],CFG.DEFAULT_POST)
        msg += f" | official F1 {q:.4f}"
        history.append(dict(epoch=epoch,loss=ep_loss,f1=q))
        if q>best_f1:
            best_f1,best_epoch=q,epoch
            torch.save({
                "state_dict":model.state_dict(),
                "encoder":ACTIVE_ENCODER,
                "family":ACTIVE_FAMILY,
                "epoch":epoch,"f1":q
            },BEST_PATH)
            msg += "  *"
    print(msg,"|",elapsed(),flush=True)

assert BEST_PATH.exists(), "No checkpoint was saved."
pd.DataFrame(history).to_csv(TRAIN_LOG,index=False)
print("\nBEST:",best_epoch,best_f1,BEST_PATH)


ep 01/55 loss 1.5963 | 0h 05m 36s
ep 02/55 loss 1.4142 | 0h 05m 46s
ep 03/55 loss 1.3295 | 0h 05m 56s
ep 04/55 loss 1.2772 | 0h 06m 11s
ep 05/55 loss 1.2168 | 0h 06m 26s
ep 06/55 loss 1.1789 | 0h 06m 42s
ep 07/55 loss 1.1267 | 0h 06m 59s
ep 08/55 loss 1.0909 | 0h 07m 16s
ep 09/55 loss 1.0761 | 0h 07m 35s
ep 10/55 loss 1.0112 | 0h 07m 52s
ep 11/55 loss 0.9899 | 0h 08m 09s
ep 12/55 loss 0.9442 | 0h 08m 26s
ep 13/55 loss 0.9196 | 0h 08m 44s
ep 14/55 loss 0.9030 | 0h 09m 01s
ep 15/55 loss 0.8593 | official F1 0.7330  * | 0h 09m 28s
ep 16/55 loss 0.8423 | 0h 09m 45s
ep 17/55 loss 0.8163 | 0h 10m 02s
ep 18/55 loss 0.7965 | 0h 10m 19s
ep 19/55 loss 0.7874 | 0h 10m 37s
ep 20/55 loss 0.7661 | official F1 0.7628  * | 0h 11m 02s
ep 21/55 loss 0.7464 | 0h 11m 20s
ep 22/55 loss 0.7473 | 0h 11m 38s
ep 23/55 loss 0.7226 | 0h 11m 55s
ep 24/55 loss 0.7066 | 0h 12m 12s
ep 25/55 loss 0.6996 | official F1 0.7312 | 0h 12m 37s
ep 26/55 loss 0.7005 | 0h 12m 55s
ep 27/55 loss 0.6824 | 0h 13m 12s
ep 28/55 loss

In [13]:

# 11 · Reload best + 4-flip TTA maps
blob=torch.load(BEST_PATH,map_location="cpu")
best_model=Stage13DINO(ACTIVE_ENCODER,pretrained=False).to(DEVICE)
best_model.load_state_dict(blob["state_dict"])
best_model.eval()

del model,opt,scaler,loader
gc.collect(); torch.cuda.empty_cache()

VAL_MAPS={}
for j,image_id in enumerate(val_ids,1):
    VAL_MAPS[image_id]=predict_one(best_model,IMAGES[image_id],tta=CFG.TTA_FLIPS)
    if j%15==0: print("val TTA",j,"/",len(val_ids),elapsed())

TEST_MAPS={}
for j,image_id in enumerate(TEST_IDS,1):
    gray=np.array(Image.open(TEST_DIR/f"{image_id}.png").convert("L"),np.uint8)
    TEST_MAPS[image_id]=predict_one(best_model,gray,tta=CFG.TTA_FLIPS)
    if j%30==0: print("test TTA",j,"/",len(TEST_IDS),elapsed())

np.savez_compressed(MAPS/f"{CFG.RUN_TAG}_val.npz",
                    **{k:(np.clip(v,0,1)*255).astype(np.uint8) for k,v in VAL_MAPS.items()})
np.savez_compressed(MAPS/f"{CFG.RUN_TAG}_test.npz",
                    **{k:(np.clip(v,0,1)*255).astype(np.uint8) for k,v in TEST_MAPS.items()})
print("maps saved")


DINO intermediate shapes: [(1, 384, 16, 16), (1, 384, 16, 16), (1, 384, 16, 16), (1, 384, 16, 16)]
val TTA 15 / 85 0h 22m 13s
val TTA 30 / 85 0h 22m 16s
val TTA 45 / 85 0h 22m 20s
val TTA 60 / 85 0h 22m 23s
val TTA 75 / 85 0h 22m 26s
test TTA 30 / 182 0h 22m 35s
test TTA 60 / 182 0h 22m 42s
test TTA 90 / 182 0h 22m 49s
test TTA 120 / 182 0h 22m 56s
test TTA 150 / 182 0h 23m 02s
test TTA 180 / 182 0h 23m 09s
maps saved


In [14]:

# 12 · Full validation sweep
rows=[]
per_image={}
for tc in CFG.TC_GRID:
    for ts in CFG.TS_GRID:
        decoded={}
        for image_id in val_ids:
            fg,seed=VAL_MAPS[image_id]
            cell=fg>=tc
            markers,nm=ndi.label(seed>=ts)
            if nm==0 or not cell.any():
                decoded[image_id]=[]
            else:
                lab=watershed(-fg,markers,mask=cell)
                decoded[image_id]=[
                    np.flatnonzero(lab.reshape(-1)==k).astype(np.int32)
                    for k in range(1,int(lab.max())+1)
                ]
        for ma in CFG.MA_GRID:
            vals=[]; counts=[]
            for image_id in val_ids:
                pred=[f for f in decoded[image_id] if len(f)>=ma]
                vals.append(instance_f1(INST[image_id],pred))
                counts.append(len(pred))
            key=(tc,ts,ma)
            per_image[key]=np.array(vals)
            rows.append(dict(t_cell=tc,t_seed=ts,min_area=ma,
                             f1=float(np.mean(vals)),mean_pred=float(np.mean(counts))))

SWEEP=pd.DataFrame(rows).sort_values("f1",ascending=False).reset_index(drop=True)
print(SWEEP.head(12).to_string(index=False))
peak=float(SWEEP.iloc[0].f1)
plateau=SWEEP[SWEEP.f1>=peak-CFG.PLATEAU_TOL]
def snap(v,grid): return min(grid,key=lambda x:abs(x-v))
POST=dict(
    t_cell=snap(float(np.median(plateau.t_cell)),CFG.TC_GRID),
    t_seed=snap(float(np.median(plateau.t_seed)),CFG.TS_GRID),
    min_area=int(snap(float(np.median(plateau.min_area)),CFG.MA_GRID)),
    score_thr=0.0
)
KEY=(POST["t_cell"],POST["t_seed"],POST["min_area"])
VAL_PER_IMAGE=per_image[KEY]
VAL_F1=float(VAL_PER_IMAGE.mean())
print("\npeak:",round(peak,4),"| plateau:",POST,"| F1:",round(VAL_F1,4))
print("Stage 10 reference: local 0.8072 | public 0.5295")


 t_cell  t_seed  min_area       f1  mean_pred
   0.50    0.70       110 0.798225  98.176471
   0.45    0.70        90 0.797568 103.082353
   0.60    0.75        90 0.797544 101.623529
   0.45    0.70       110 0.797320  99.105882
   0.50    0.65       110 0.797302  97.600000
   0.40    0.70       110 0.797268  99.847059
   0.50    0.75       110 0.797225  98.729412
   0.60    0.60        90 0.797187  99.905882
   0.50    0.60       110 0.797136  97.352941
   0.55    0.70       110 0.797135  97.270588
   0.55    0.60       110 0.797109  96.529412
   0.60    0.70        90 0.797067 101.047059

peak: 0.7982 | plateau: {'t_cell': 0.5, 't_seed': 0.65, 'min_area': 90, 'score_thr': 0.0} | F1: 0.7956
Stage 10 reference: local 0.8072 | public 0.5295


In [15]:

# 13 · Density audit
audit=[]
for image_id,f1 in zip(val_ids,VAL_PER_IMAGE):
    pred=instances_from_maps(VAL_MAPS[image_id][0],VAL_MAPS[image_id][1],POST)
    ng,npred=len(INST[image_id]),len(pred)
    tp=f1*(ng+npred)/2
    audit.append(dict(id=image_id,n_gt=ng,n_pred=npred,f1=float(f1),fn=ng-tp,fp=npred-tp))

AUDIT=pd.DataFrame(audit)
edges=[0,50,200,400,10**9]
labels=["sparse <50","medium 50-199","dense 200-399","very dense >=400"]
AUDIT["bin"]=pd.cut(AUDIT.n_gt,edges,labels=labels,right=False)
DENSITY=AUDIT.groupby("bin",observed=False).agg(
    images=("id","size"),mean_f1=("f1","mean"),mean_gt=("n_gt","mean"),
    mean_pred=("n_pred","mean"),mean_fn=("fn","mean"),mean_fp=("fp","mean")
)
DENSITY["pred_over_gt"]=DENSITY.mean_pred/DENSITY.mean_gt
print(DENSITY.round(4).to_string())
print("\nStage 10 pred/GT reference:")
print("sparse 0.9484 | medium 0.9860 | dense 0.9036 | very dense 0.8703")


                  images  mean_f1   mean_gt  mean_pred  mean_fn  mean_fp  pred_over_gt
bin                                                                                   
sparse <50            49   0.8266   27.8163    25.9184   5.4082   3.5102        0.9318
medium 50-199         23   0.7380   97.0000   102.6957  22.9565  28.6522        1.0587
dense 200-399          7   0.7508  295.0000   306.1429  68.2857  79.4286        1.0378
very dense >=400       6   0.8158  499.1667   481.0000  98.8333  80.6667        0.9636

Stage 10 pred/GT reference:
sparse 0.9484 | medium 0.9860 | dense 0.9036 | very dense 0.8703


In [16]:

# 14 · RLE parity + 13th_submission.csv
parity=[]
for image_id in val_ids[:25]:
    pred=instances_from_maps(VAL_MAPS[image_id][0],VAL_MAPS[image_id][1],POST)
    rt=[rle_decode_flat(flat_to_rle(f),CFG.IMG_H,CFG.IMG_W,"C") for f in pred]
    parity.append(instance_f1(INST[image_id],rt))
direct=float(np.mean([VAL_PER_IMAGE[val_ids.index(i)] for i in val_ids[:25]]))
delta=abs(direct-float(np.mean(parity)))
print("RLE parity delta:",delta)
assert delta<0.002

SUB_PATH=Path("/content/13th_submission.csv")
rows=[]; test_counts=[]; fallbacks=0
for image_id in TEST_IDS:
    pred=instances_from_maps(TEST_MAPS[image_id][0],TEST_MAPS[image_id][1],POST)
    if not pred:
        fallbacks+=1
        pred=instances_from_maps(
            TEST_MAPS[image_id][0],TEST_MAPS[image_id][1],
            dict(t_cell=0.30,t_seed=0.30,min_area=1,score_thr=0.0)
        )[:1]
    if not pred:
        y,x=CFG.IMG_H//2,CFG.IMG_W//2
        pred=[np.array([y*CFG.IMG_W+x],np.int32)]
    test_counts.append(len(pred))
    rows.extend((image_id,flat_to_rle(f)) for f in pred)

sub=pd.DataFrame(rows,columns=[ID_COL,ANN_COL])
sub.insert(0,CFG.SUB_INDEX_COL,np.arange(len(sub),dtype=np.int64))
sub.to_csv(SUB_PATH,index=False)
assert set(sub[ID_COL].astype(str))==set(TEST_IDS)
print("wrote:",SUB_PATH)
print("pred/image mean/min/max:",
      round(float(np.mean(test_counts)),1),int(np.min(test_counts)),int(np.max(test_counts)),
      "| fallbacks:",fallbacks)
if not CFG.SUBMISSION_RULES_CONFIRMED:
    print("!!! Confirm DINOv3 generic LVD-1689M pretraining with the TA before submitting.")


RLE parity delta: 0.0
wrote: /content/13th_submission.csv
pred/image mean/min/max: 106.6 8 630 | fallbacks: 0
!!! Confirm DINOv3 generic LVD-1689M pretraining with the TA before submitting.


In [17]:

# 15 · Report + artifact preservation + downloads
REPORT_PATH=Path("/content/13th_REPORT.md")
DRIVE_SUB=ART/"13th_submission.csv"
DRIVE_REPORT=ART/"13th_REPORT.md"
shutil.copy2(SUB_PATH,DRIVE_SUB)
shutil.copy2(BEST_PATH,ART/f"{CFG.RUN_TAG}_selected_model.pt")

dense_text="\n".join("    "+s for s in DENSITY.round(4).to_string().splitlines())
report_lines=[
    f"# Stage 13 report — {ACTIVE_FAMILY} dense cell segmentation",
    "",
    f"- active encoder: {ACTIVE_ENCODER}",
    "- intended primary: DINOv3 ViT-S/16 pretrained on generic LVD-1689M web imagery",
    "- architecture: DINO intermediate fusion + shallow CNN detail branch + progressive decoder",
    "- output: foreground + relative e70 seed maps -> marker-controlled watershed",
    f"- grouped split: fold {CFG.FOLD}/{CFG.N_FOLDS} | groups {STATS['groups']} | train/val {STATS['n_train']}/{STATS['n_val']}",
    f"- max val-to-train thumbnail correlation: {STATS['max_val_train_corr']:.3f}",
    f"- checkpoint epoch: {blob['epoch']} | quick official F1: {blob['f1']:.4f}",
    f"- final TTA swept F1: {VAL_F1:.4f} (peak {peak:.4f})",
    f"- chosen post-processing: {POST}",
    f"- test predictions/image: mean {np.mean(test_counts):.1f} (min {np.min(test_counts)}, max {np.max(test_counts)})",
    f"- fallbacks: {fallbacks}",
    f"- RLE parity delta: {delta:.6f}",
    f"- runtime: {elapsed()}",
    "",
    "## Density validation",
    dense_text,
    "",
    "## Stage 10 reference",
    "- local plateau F1: 0.8072",
    "- public: 0.5295",
    "- pred/GT: sparse 0.9484, medium 0.9860, dense 0.9036, very dense 0.8703",
    "",
    "## Rules",
    "Generic pretrained weights are allowed per the known guidance, but DINOv3 LVD-1689M was not specifically TA-confirmed like ImageNet/COCO.",
    f"SUBMISSION_RULES_CONFIRMED = {CFG.SUBMISSION_RULES_CONFIRMED}",
    "Do not submit this candidate until the TA confirms that this generic DINOv3 pretrained source is acceptable.",
    "",
    "## Outputs",
    "- /content/13th_submission.csv",
    "- /content/13th_REPORT.md",
    f"- Drive artifacts: {ART}",
]
report="\n".join(report_lines)
REPORT_PATH.write_text(report)
shutil.copy2(REPORT_PATH,DRIVE_REPORT)
print(report)

print("\nSuggested Kaggle description after rules confirmation:")
print(f"Y_13th: {ACTIVE_FAMILY} ViT-S dense segmentation + CNN detail decoder, e70 foreground/seed watershed, density-focused D4/scale augmentation, grouped-val F1 {VAL_F1:.4f}, no ensemble.")
print("\nCHECKPOINT PRESERVED:",ART/f"{CFG.RUN_TAG}_selected_model.pt")
print("submission:",SUB_PATH)
print("report:",REPORT_PATH)

try:
    from google.colab import files
    files.download(str(SUB_PATH))
    files.download(str(REPORT_PATH))
except Exception as e:
    print("Auto-download skipped:",e)


# Stage 13 report — DINOv3 dense cell segmentation

- active encoder: vit_small_patch16_dinov3.lvd1689m
- intended primary: DINOv3 ViT-S/16 pretrained on generic LVD-1689M web imagery
- architecture: DINO intermediate fusion + shallow CNN detail branch + progressive decoder
- output: foreground + relative e70 seed maps -> marker-controlled watershed
- grouped split: fold 0/5 | groups 424 | train/val 339/85
- max val-to-train thumbnail correlation: 0.666
- checkpoint epoch: 50 | quick official F1: 0.7935
- final TTA swept F1: 0.7956 (peak 0.7982)
- chosen post-processing: {'t_cell': 0.5, 't_seed': 0.65, 'min_area': 90, 'score_thr': 0.0}
- test predictions/image: mean 106.6 (min 8, max 630)
- fallbacks: 0
- RLE parity delta: 0.000000
- runtime: 0h 25m 36s

## Density validation
                      images  mean_f1   mean_gt  mean_pred  mean_fn  mean_fp  pred_over_gt
    bin                                                                                   
    sparse <50            49   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>